In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
df = pd.read_csv('../data/raw/application_train.csv')
print(f"Data Loaded: {df.shape}")

Data Loaded: (307511, 122)


In [2]:
# Ratio Feature 
# Business rationale documented for each feature

# 1. Debt to Income ratio
# How much total debt is the borrower taking on relative to annual income ?
# Higher Ratio = Less repayment buffer = higher default risk

df['DEBT_TO_INCOME'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

# 2. Annuity to income
# What % of income goes towards loan repayment?
# Industry rule of thumb. Above 40% is high risk
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'] 

# 3. Credit to Goods ratio
# How much of the purchase is being financed vs paid upfront?
# Ratio of 1.0 means 100% financed - no borrower equity
df['CREDIT_TO_GOODS'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']

# 4. Annuity to Credit Ration
# Implied loan tenure signal - lower ratio means longer repayment period
# Longer Tenure = more exposure to life event that cause defaulter 
df['ANNUITY_TO_CREDIT'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

# 5. Income per family member 
# Disposable income adjusted for dependents
df['INCOME_PER_PERSON'] = (df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0 , np.nan))

# 6. External Source Mean Score
# Average of three external credit score - reduce the noice of missingness
# More stable single signal than any individual score 
df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1' , 'EXT_SOURCE_2' , 'EXT_SOURCE_3']].mean(axis=1)

# 7. External Source Weighted Mean
# Ext - 3 showed highest correlation - give it more weight 
df['EXT_SOURCE_WEIGHTED'] = (
    0.5 * df['EXT_SOURCE_3'].fillna(df['EXT_SOURCE_3'].median()) + 
    0.3 * df['EXT_SOURCE_2'].fillna(df['EXT_SOURCE_2'].median()) +
    0.2 * df['EXT_SOURCE_1'].fillna(df['EXT_SOURCE_1'].median())
)

# Replace any infinity values with NaN across all new ratio features
ratio_cols = ['DEBT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_GOODS',
              'ANNUITY_TO_CREDIT', 'INCOME_PER_PERSON',
              'EXT_SOURCE_MEAN', 'EXT_SOURCE_WEIGHTED']



print("Ratio features created successfully.")
print("\n New column added : 7")
print(f"\n Total column : {df.shape[1]}")

# Check for infinity first
for col in ratio_cols:
    inf_count = np.isinf(df[col]).sum()
    if inf_count > 0:
        print(f"{col}: {inf_count} infinity values found — replacing with NaN")
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

print("\nInfinity check complete")
df['DEBT_TO_INCOME'].head()

Ratio features created successfully.

 New column added : 7

 Total column : 129

Infinity check complete


0    2.007889
1    4.790750
2    2.000000
3    2.316167
4    4.222222
Name: DEBT_TO_INCOME, dtype: float64

In [3]:
# Validate ratio features — do they outperform raw columns?
raw_cols = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 
            'AMT_ANNUITY', 'AMT_GOODS_PRICE',
            'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

new_cols = ['DEBT_TO_INCOME', 'ANNUITY_TO_INCOME',
            'CREDIT_TO_GOODS', 'ANNUITY_TO_CREDIT',
            'INCOME_PER_PERSON', 'EXT_SOURCE_MEAN',
            'EXT_SOURCE_WEIGHTED']

compare_cols = raw_cols + new_cols

correlations = df[compare_cols].corrwith(df['TARGET']).abs()
correlations = correlations.sort_values(ascending=False)

print("=== RAW vs ENGINEERED FEATURE CORRELATION ===\n")
for col in correlations.index:
    tag = " ← ENGINEERED" if col in new_cols else ""
    print(f"{col:<35} {correlations[col]:.4f}{tag}")

=== RAW vs ENGINEERED FEATURE CORRELATION ===

EXT_SOURCE_MEAN                     0.2221 ← ENGINEERED
EXT_SOURCE_WEIGHTED                 0.2192 ← ENGINEERED
EXT_SOURCE_3                        0.1789
EXT_SOURCE_2                        0.1605
EXT_SOURCE_1                        0.1553
CREDIT_TO_GOODS                     0.0694 ← ENGINEERED
AMT_GOODS_PRICE                     0.0396
AMT_CREDIT                          0.0304
ANNUITY_TO_INCOME                   0.0143 ← ENGINEERED
AMT_ANNUITY                         0.0128
ANNUITY_TO_CREDIT                   0.0127 ← ENGINEERED
DEBT_TO_INCOME                      0.0077 ← ENGINEERED
INCOME_PER_PERSON                   0.0066 ← ENGINEERED
AMT_INCOME_TOTAL                    0.0040


In [4]:
# Check default rate by DTI bands — reveals non-linear relationship
df['DTI_BAND'] = pd.cut(df['DEBT_TO_INCOME'],
                         bins=[0, 1, 2, 3, 4, 5, 100],
                         labels=['0-1x','1-2x','2-3x',
                                 '3-4x','4-5x','5x+'])

print("=== DEFAULT RATE BY DEBT-TO-INCOME BAND ===")
dti_default = df.groupby('DTI_BAND')['TARGET'].agg(['mean','count'])
dti_default.columns = ['Default Rate %', 'Count']
dti_default['Default Rate %'] = (dti_default['Default Rate %'] * 100).round(2)
print(dti_default.to_string())

=== DEFAULT RATE BY DEBT-TO-INCOME BAND ===
          Default Rate %  Count
DTI_BAND                       
0-1x                6.44  16174
1-2x                7.70  60164
2-3x                8.69  64858
3-4x                9.11  48703
4-5x                8.47  36805
5x+                 7.38  80807
